# Svara (स्वर) — Two-Way Intelligent AAC Speech Assistant

Svara is a next-generation **Augmentative and Alternative Communication (AAC)** assistant that enables non-verbal individuals (such as people living with cerebral palsy, autism, post-stroke aphasia, or laryngectomy) to engage in natural, two-way Indian-language conversations.

### Complete Assistive AI Architecture:
1. **Hearing (ASR):** Caregiver or doctor speech is transcribed using Bodhan's `indic-transcribe`.
2. **Reasoning (Intent & Context):** Understands caregiver questions to generate 3–4 instant, 1-tap reply options.
3. **Telegraphic Concept Expansion:** Non-verbal users tap 2–3 concepts (`[Water]`, `[Cold]`, `[Want]`), which are assembled into natural code-mixed sentences.
4. **Voice (Personalised TTS):** Vocalises speech in the user's chosen acoustic identity using Bodhan's `indic-speak`.
5. **Zero-Latency Audio Cache:** Caches high-frequency phrases locally for instantaneous (0 ms) offline playback.

**Bodhan Models:** `indic-transcribe`, `indic-speak`

**Requires:** `BODHAN_API_KEY` in `.env` (see `.env.example`).

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os, json, time
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from svara import SvaraClient

# Load credentials
load_dotenv(find_dotenv())
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

client = SvaraClient()

## 1. Hearing the Doctor/Caregiver (`indic-transcribe`)

A caregiver or physician asks a question. We transcribe the incoming spoken audio using Bodhan's `indic-transcribe` model.

In [ ]:
sample_audio = Path("../../sample_data/sample_hindi.wav")

if sample_audio.exists():
    transcript, dt_asr = client.transcribe(sample_audio, lang="hi")
    print(f"Caregiver Spoke ({dt_asr*1000:.0f} ms):\n\"{transcript}\"")
else:
    transcript = "Kya aapko paani chahiye ya fresh juice mangwa doon?"
    print(f"Using mock transcript:\n\"{transcript}\"")

## 2. Generating Contextual 1-Tap Replies

Instead of making motor-impaired individuals type long responses, Svara reasons over the caregiver's question to generate context-specific 1-tap reply options.

In [ ]:
caregiver_question = "Kya aapko paani chahiye ya fresh juice mangwa doon?"
replies, dt_reason = client.suggest_contextual_replies(caregiver_question, lang="hi")

print(f"Contextual Reasoning ({dt_reason*1000:.0f} ms):")
for i, r in enumerate(replies, 1):
    print(f"  {i}. {r}")

## 3. Telegraphic Concept-to-Speech Expansion

When composing novel thoughts, users tap 2–3 concept tokens (e.g. `[Water]`, `[Cold]`, `[Want]`). Svara reconstructs a fluent, polite code-mixed sentence.

In [ ]:
tokens = ["Paani (Water)", "Thanda (Cold)", "Chahiye (Want)"]
expanded_sentence, dt_exp = client.expand_telegraphic_tokens(tokens, lang="hi")
print(f"Tokens:    {tokens}")
print(f"Expanded:  \"{expanded_sentence}\" ({dt_exp*1000:.0f} ms)")

## 4. Voice Personalisation (`indic-speak`)

Non-verbal communicators deserve an acoustic persona that matches their age and identity. We vocalise the response across available voices.

In [ ]:
phrase = expanded_sentence or "Mujhe thoda thanda paani chahiye, please."
voices = ["Kavya", "Amit", "Suhani"]

for v in voices:
    audio, is_cached, dt = client.synthesize(phrase, voice=v, lang="hi", use_cache=False)
    p = OUT / f"voice_{v}.wav"
    p.write_bytes(audio)
    print(f"{v:10} -> {p.name} ({len(audio) // 1024} KB in {dt*1000:.0f} ms)")

## 5. Zero-Latency AAC Audio Caching Benchmark

In conversation, waiting 1.5–2 seconds creates conversational strain. Svara pre-caches frequently used phrases with deterministic SHA-256 keys for **0 ms** network latency.

In [ ]:
test_phrase = "Emergency hai, jaldi aao please."

# Call 1: Live API Request
_, cached_1, latency_cold = client.synthesize(test_phrase, voice="Kavya", lang="hi", use_cache=True)
print(f"Cold Call (Live API):    {latency_cold*1000:.1f} ms (cached: {cached_1})")

# Call 2: Local Disk Cache
audio_cached, cached_2, latency_warm = client.synthesize(test_phrase, voice="Kavya", lang="hi", use_cache=True)
print(f"Warm Call (Local Cache): {latency_warm*1000:.1f} ms (cached: {cached_2})")

reduction = (1 - (latency_warm / max(latency_cold, 0.001))) * 100
print(f"Latency Reduction:       {reduction:.1f}%")

## 6. Interactive Web Application

To launch the touch-accessible Two-Way communication board, run:

```bash
streamlit run app.py
```